# Перезапуск полного IgBLAST для ERR346598–ERR346601 (4 потока)

`ERR346598` был убит SIGKILL (rc=-9) при `-num_threads 8` — OOM. Перезапускаем последовательно с `-num_threads 4`, heartbeat каждые 120 сек. Недоделанные TSV (< 10% размера фасты) удаляются и пересчитываются.


In [ ]:
from pathlib import Path
import os, subprocess, time

ENV = "/data/user/epishkin/conda/envs/bcr_env"
IGDATA = Path("/data/user/epishkin/igblast")
os.environ["IGDATA"] = str(IGDATA)
os.environ["PATH"] = str(IGDATA / "bin") + ":" + ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = ENV + "/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

BASE = Path("/data/user/epishkin/results/ERP003950")
FA_DIR = BASE / "igblast/fasta"
OUT_DIR = BASE / "igblast"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = ["ERR346598", "ERR346599", "ERR346600", "ERR346601"]
LOG = OUT_DIR / "full_igblast_retry_4threads.log"
print("PID", os.getpid(), "log:", LOG)

def log(msg):
    ts = time.strftime("%F %T")
    with LOG.open("a") as f:
        f.write(ts + " " + msg + "\n")
    print(ts, msg, flush=True)

log("START_RETRY_4THREADS")

for sample in SAMPLES:
    fa = FA_DIR / (sample + ".fa")
    if not fa.exists():
        log("SKIP " + sample + " no FASTA")
        continue
    out = OUT_DIR / (sample + "_igblast.tsv")
    lg = OUT_DIR / (sample + "_igblast_retry.log")
    fa_mb = fa.stat().st_size / 1e6
    if out.exists():
        out_mb = out.stat().st_size / 1e6
        if out_mb < fa_mb * 0.1:
            out.unlink(missing_ok=True)
            log("DELETE stale " + sample)
        else:
            log("SKIP complete " + sample)
            continue
    n = int(subprocess.run(["grep", "-c", "^>", str(fa)], capture_output=True, text=True).stdout.strip() or 0)
    log("RUN %s n=%d fa_mb=%.1f" % (sample, n, fa_mb))
    cmd = [
        str(IGDATA / "bin/igblastn"),
        "-germline_db_V", str(IGDATA / "internal_data/mouse/mouse_gl_V"),
        "-germline_db_D", str(IGDATA / "internal_data/mouse/mouse_gl_D"),
        "-germline_db_J", str(IGDATA / "internal_data/mouse/mouse_gl_J"),
        "-organism", "mouse",
        "-ig_seqtype", "Ig",
        "-domain_system", "imgt",
        "-query", str(fa),
        "-auxiliary_data", str(IGDATA / "optional_file/mouse_gl.aux"),
        "-outfmt", "19",
        "-num_threads", "4",
        "-out", str(out),
    ]
    t0 = time.time()
    with open(lg, "w") as logfile:
        proc = subprocess.Popen(cmd, stdout=logfile, stderr=subprocess.STDOUT)
    log("PID %s %d" % (sample, proc.pid))
    while proc.poll() is None:
        elapsed = int(time.time() - t0)
        mb = out.stat().st_size / 1e6 if out.exists() else 0
        log("HEARTBEAT %s elapsed=%ds out_mb=%.1f pid=%d" % (sample, elapsed, mb, proc.pid))
        time.sleep(120)
    rc = proc.returncode
    if rc != 0:
        log("ERROR %s rc=%d" % (sample, rc))
        break
    log("DONE %s rc=0 mb=%.1f" % (sample, out.stat().st_size / 1e6))
else:
    log("ALL_DONE_RETRY_4THREADS")
print("FINISHED")


In [ ]:
# Проверка размеров выходных TSV
from pathlib import Path
p = Path("/data/user/epishkin/results/ERP003950/igblast")
for f in sorted(p.glob("ERR*_igblast.tsv")):
    print("%-35s %10.1f MB" % (f.name, f.stat().st_size / 1e6))
